In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/ECS189G_Self_Consistency/results"
import os
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

# UL2 on AQUA — Self-Consistency (k=40)

Runs `google/ul2` (20B seq2seq) with self-consistency decoding (`k=40`) on the AQUA-RAT test set.  
This model runs locally via HuggingFace Transformers — no API key needed.  
Uses the disk response cache so re-runs skip inference.

## 1. Load & preview the AQUA dataset

In [ ]:
from pipeline_data import load_benchmark_dataset
from prompts import BenchmarkType

dataset = load_benchmark_dataset(BenchmarkType.AQUA)
print(f"AQUA test set: {len(dataset)} questions")
print(f"Example: {dataset[0]['question'][:120]}...")
print(f"Gold answer: {dataset[0]['final_answer']}")

## 2. Verify model loads

UL2 is ~20B parameters. On Colab, this requires a GPU runtime (A100 recommended).  
The wrapper will attempt 4-bit quantization → float16 → CPU fallback automatically.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Quick smoke test
from model_wrapper import run_model
test_out = run_model("google/ul2", "Q: What is 2 + 2?\nA:", temperature=0.0, num_samples=1)
print(f"Smoke test output: {test_out}")

## 3. Run evaluation

In [ ]:
from pipeline_eval import evaluate

results = evaluate(
    model="google/ul2",
    benchmark=BenchmarkType.AQUA,
    dataset=dataset,
    method="self_consistency",
    k=40,
    self_consistency_temperature=0.7,
)

print(f"\n=== Results ===")
print(f"Model:    {results.model}")
print(f"Dataset:  {results.dataset}")
print(f"Method:   {results.method}")
print(f"k:        {results.k}")
print(f"Accuracy: {results.accuracy:.4f} ({results.correct}/{results.total})")
print(f"Parsed:   {results.parsed}/{results.total}")

## 4. Save results to CSV

In [ ]:
from pathlib import Path
from pipeline_io import write_results

# Save locally
local_path = Path("results/aqua_k40_ul2.csv")
write_results([results], output_csv=local_path)
print(f"Saved locally to {local_path}")

# Save to Google Drive
drive_path = Path(DRIVE_RESULTS_DIR) / "aqua_k40_ul2.csv"
write_results([results], output_csv=drive_path)
print(f"Saved to Google Drive: {drive_path}")

## 5. Inspect failures

In [ ]:
failures = [d for d in results.details if not d["is_correct"]]
print(f"{len(failures)} incorrect out of {results.total}\n")

for f in failures[:5]:
    print(f"Q: {f['question'][:100]}...")
    print(f"  Gold: {f['gold_answer']}  |  Predicted: {f['predicted']}")
    print()